# Анализ аудитории витуберов

Посмотрим ключевую статистику комьюнити витуберов:

**Что сделано**:

* Рост аудитории
* Уникальность витуберов
* Кластеризация витуберов
* Количество локальных фолловов у зрителей выборки
* Количество глобальных фолловов у зрителей выборки
* Граф самых близости витуберов (то есть если косинусное расстояние меньше какого-то порога, то рисовать ребро, иначе - нет)

**Что пока нет**:

* Размер активной аудитории
* Активность чата
* Кластеризация сообщений в чате

#### Импорт всего нужного

In [1]:
import os
import csv
import sys
sys.path.insert(1, '../util/')

from data import UserData, FollowerData, json_to_user_data
from userdata import get_userdata, get_userdata_by_login
from followers import get_followers

import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import numpy as np
import sklearn
from datetime import datetime, timedelta
from typing import Optional, List, Tuple, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

## Prepare data

In [2]:
VTUBERS_LIST_FILE_PATH = "./data/vtubers.txt"

In [3]:
@dataclass
class VtubersData:
    vtuber_names: List[str] # 1 x N
    vtuber_followers: Dict[str, List[FollowerData]]
    user_following: Dict[int, List[str]]

In [5]:
filelines = []
vtuber_names = set()

with open(VTUBERS_LIST_FILE_PATH, 'r') as data_file:
    counter = 0
    
    for fileline in data_file.readlines():
        counter += 1
        print(f"Reading line #{counter}")
        fileline = fileline.strip()
        if '#' in fileline:
            line, comment = fileline.split('#', 1)
        else:
            line, comment = fileline, ""
        
        line = line.strip()
        if line:  # Got vtuber name or vtuber id
            try:
                vtuber_id = int(line)
                vtuber_name = None
            except ValueError:
                vtuber_id = None
                vtuber_name = line
            
            if vtuber_id is not None:
                vtuber_data = get_userdata(id=vtuber_id, log=True)
                
                if vtuber_data is None:
                    print(f"WARNING: no vtuber with id {vtuber_id} found:", fileline)
                    filelines.append(fileline)
                else:
                    if comment.strip().lower() != vtuber_data.login.lower():
                        new_fileline = f"{vtuber_data.id} # {vtuber_data.login.lower()}"
                        print("WARNING: should update line:", fileline, "-->", new_fileline)
                        filelines.append(new_fileline)
                    else:
                        filelines.append(fileline)
                    vtuber_names.add(vtuber_data.login)
            else:
                vtuber_data = get_userdata_by_login(login=vtuber_name, log=True)

                if vtuber_data is None:
                    print(f"WARNING: no vtuber with name {vtuber_name} found:", fileline)
                    filelines.append(fileline)
                else:
                    new_fileline = f"{vtuber_data.id} # {vtuber_name}"
                    print("WARNING: should update line:", fileline, "-->", new_fileline)
                    vtuber_names.add(vtuber_data.login)
                    filelines.append(new_fileline)
        else: # Got empty line or comment line
            filelines.append(fileline)

vtuber_names = list(vtuber_names)

Reading line #1
Reading line #2
Reading line #3
Reading line #4
Reading line #5
Reading line #6
Reading line #7
Reading line #8
Reading line #9
Reading line #10
Reading line #11
Reading line #12
Reading line #13
Reading line #14
Reading line #15
Reading line #16
Reading line #17
Reading line #18
Reading line #19
Reading line #20
Reading line #21
Reading line #22
Reading line #23
Reading line #24
Reading line #25
Reading line #26
Reading line #27
Reading line #28
Reading line #29
Reading line #30
Reading line #31
Reading line #32
Reading line #33
Reading line #34
Reading line #35
Reading line #36
Reading line #37
Reading line #38
Reading line #39
Reading line #40
Reading line #41
Reading line #42
Reading line #43
Reading line #44
Reading line #45
Reading line #46
Reading line #47
Reading line #48
Reading line #49
Reading line #50
Reading line #51
Reading line #52
Reading line #53
Reading line #54
Reading line #55
Reading line #56
Reading line #57
Reading line #58
Reading line #59
Readin

In [6]:
# Run if you think you should update vtubers.txt file
with open(VTUBERS_LIST_FILE_PATH, 'w') as data_file:
    for fileline in filelines:
        data_file.write(fileline)
        data_file.write("\n")

In [7]:
print(f"Loaded {len(vtuber_names)} vtubers")

Loaded 269 vtubers


In [8]:
def get_cache_path():
    current_date = datetime.now()

    return os.path.join("data/", 
                        ".temp/",
                        f"followers_{current_date.year}_{current_date.month}_{current_date.day}.txt")


def load_followers_data(vtuber_names: List[str],
                        online: bool = False, 
                        num_workers: int = 32,
                        cache_path: str = get_cache_path()):
    if online:
        def load_followers_execution(vtuber_login: str) -> Tuple[str, Optional[List[FollowerData]]]:
            result = get_followers(vtuber_login, log=True, repeat_times=5, repeat_delay=1.0)
            if result is None:
                print("[ERROR]", "Failed loading vtuber", vtuber_login, "attempt", i)
            else:
                return vtuber_login, result


        vtuber_followers = {}
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = []

            for vtuber in vtuber_names:
                future = executor.submit(load_followers_execution, vtuber_login=vtuber)
                futures.append(future)
            
            for future in as_completed(futures):
                vtuber_login, followers_result = future.result()
                if followers_result is None:
                    print(f"Cannot load `{vtuber_login}` result")
                else:
                    empty_results = list(filter(lambda el: el.user is None, followers_result))
                    print(f"Got `{vtuber_login}` result, empty results: {len(empty_results)}/{len(followers_result)}")
                vtuber_followers[vtuber_login] = followers_result
        
        return vtuber_followers
    else:
        result = {}

        with open(cache_path, 'r') as cache_file:
            csvreader = csv.DictReader(cache_file, delimiter=',')

            for row in csvreader:
                vtuber = row['vtuber']
                followed_at = datetime.fromisoformat(row['followed_at'])
                if row['id']:
                    id              = int(row['id'])
                    login           = None if row['login'] == "None" else row['login']
                    created_at      = None if row['created_at'] == "None" else datetime.fromisoformat(row['created_at'])
                    deleted_at      = None if row['deleted_at'] == "None" else datetime.fromisoformat(row['deleted_at'])
                    follows_count   = int(row['follows_count'])
                    user_data = UserData(id=id,
                                        login=login,
                                        created_at=created_at,
                                        deleted_at=deleted_at,
                                        follows_count=follows_count)
                else:
                    user_data = None
                follower_data = FollowerData(user=user_data,
                                            followed_at=followed_at)
                
                if vtuber not in result.keys():
                    result[vtuber] = []
                result[vtuber].append(follower_data)
        
        return result


def cache_followers_data(vtuber_followers: Dict[str, List[FollowerData]],
                        cache_path: str = get_cache_path()):
    with open(cache_path, 'w') as cache_file:
        csvwriter = csv.writer(cache_file, delimiter=',')
        csvwriter.writerow(["vtuber",
                            "followed_at",
                            "id",
                            "login",
                            "created_at",
                            "deleted_at",
                            "follows_count"])

        for (vtuber, value) in vtuber_followers.items():
            for follower_data in value:
                if follower_data.user is None:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        "",
                                        "",
                                        "",
                                        "",
                                        ""])
                else:
                    csvwriter.writerow([vtuber,
                                        follower_data.followed_at,
                                        follower_data.user.id,
                                        follower_data.user.login,
                                        str(follower_data.user.created_at),
                                        str(follower_data.user.deleted_at),
                                        follower_data.user.follows_count])

In [ ]:
vtuber_followers = load_followers_data(vtuber_names, online=True, num_workers=64)

In [ ]:
for vtuber in vtuber_names:
    assert vtuber_followers[vtuber] is not None

In [ ]:
cache_followers_data(vtuber_followers)

In [ ]:
user_following = {}

for vtuber in vtuber_names:
    for follower in vtuber_followers[vtuber]:
        if follower.user is not None:
            user_name = follower.user.id
            if user_name not in user_following.keys():
                user_following[user_name] = []
            user_following[user_name].append(vtuber)

In [ ]:
data = VtubersData(vtuber_names=vtuber_names, vtuber_followers=vtuber_followers, user_following=user_following)

## Графики роста аудитории

### График фолловов

In [ ]:
import plotly.io as pio
pio.renderers

In [ ]:
sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)


df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df, x="x", range_y=(0, 27000))
fig.show()

### График фолловеров без причин резких скачков

In [ ]:
# IGNORED_VTUBERS = ['xkamysh', "trixie_vox", "lerritay", "kepa_mita", "vernirra", "yumekomoore", "sati_akura", "planyach"]
IGNORED_VTUBERS = ['xkamysh', 'qchaan_9', 'trixie_vox', 'kepa_mita', 'lerritay']
# IGNORED_VTUBERS = ['kepa_mita']

sunce_date = datetime.fromisoformat("2023-10-01 00:00:00.000000+00:00")
# sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
auditory_data = []
for vtuber in data.vtuber_names:
    if vtuber in IGNORED_VTUBERS:
        continue
    
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        auditory_data.append(follower)

auditory_data = list(map(lambda viewer: viewer.followed_at, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = list(filter(lambda days: days > 0, auditory_data))
auditory_data = np.array(auditory_data)

df = pd.DataFrame({
    "x": auditory_data,
})
# fig = px.histogram(df, x="x", range_y=(0, 27000))
fig = px.histogram(df, x="x", cumulative=False)
fig.show() 

In [ ]:
watching_date = sunce_date + timedelta(days=394)
watching_date

In [ ]:
watching_followers = []
watching_followers_vtubers = {}
for vtuber in data.vtuber_names:
    if vtuber in IGNORED_VTUBERS:
        continue
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.followed_at.year == watching_date.year and \
                follower.followed_at.month == watching_date.month and \
                follower.followed_at.day == watching_date.day:
            watching_followers.append((vtuber, follower))
            watching_followers_vtubers[vtuber] = watching_followers_vtubers.get(vtuber, 0) + 1

In [ ]:
list(sorted(watching_followers_vtubers.items(), key=lambda el: el[1], reverse=True))[:10]

### Рост уникальных фолловов

То есть не учитываются второй, третий и т.д. фолловы от одного и того же человека

In [ ]:
IGNORED_VTUBERS = ['xkamysh', 'qchaan_9', 'trixie_vox', 'kepa_mita', 'lerritay']

sunce_date = datetime.fromisoformat("2023-11-01 00:00:00.000000+00:00")
auditory_data = {}
for vtuber in data.vtuber_names:
    if vtuber in IGNORED_VTUBERS:
        continue
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.user is not None:
            if follower.user.login not in auditory_data.keys():
                auditory_data[follower.user.login] = follower.followed_at
            auditory_data[follower.user.login] = min(auditory_data[follower.user.login], follower.followed_at)

auditory_data = auditory_data.values()
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = list(filter(lambda day: day > 0, auditory_data))
auditory_data = np.array(auditory_data)

df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df, x="x")
fig.show() 

In [ ]:
watching_date = sunce_date + timedelta(days=220)
watching_date

In [ ]:
watching_followers = []
watching_followers_vtubers = {}
for vtuber in data.vtuber_names:
    if vtuber in IGNORED_VTUBERS:
        continue
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.followed_at.year == watching_date.year and \
                follower.followed_at.month == watching_date.month and \
                follower.followed_at.day == watching_date.day:
            watching_followers.append((vtuber, follower))
            watching_followers_vtubers[vtuber] = watching_followers_vtubers.get(vtuber, 0) + 1
list(sorted(watching_followers_vtubers.items(), key=lambda el: el[1], reverse=True))[:10]

## Живая аудитория

In [ ]:
active_auditory = set()
for vtuber in data.vtuber_names:
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None and \
                follower.followed_at.year == 2025 and \
                follower.followed_at.month >= 5:
            active_auditory.add(follower.user.id)
for user in data.user_following:
    if len(data.user_following[user]) >= 2:
        active_auditory.add(user)

sunce_date = datetime.fromisoformat("2010-01-01 00:00:00.000000+00:00")
# sunce_date = datetime.fromisoformat("2024-05-01 00:00:00.000000+00:00")
until_date = datetime.fromisoformat("2025-05-01 00:00:00.000000+00:00")
# until_date = datetime.fromisoformat("2025-11-12 00:00:00.000000+00:00")
auditory_data = {}
for vtuber in data.vtuber_names:
    followers = data.vtuber_followers[vtuber]
    for follower in followers:
        if follower.user is not None and follower.user.id in active_auditory:
            if follower.user.login not in auditory_data.keys():
                auditory_data[follower.user.login] = follower.followed_at
            auditory_data[follower.user.login] = min(auditory_data[follower.user.login], follower.followed_at)

auditory_data = auditory_data.values()
auditory_data = list(filter(lambda follow_date: follow_date > sunce_date, auditory_data))
auditory_data = list(filter(lambda follow_date: follow_date < until_date, auditory_data))
auditory_data = list(map(lambda follow_date: follow_date - sunce_date, auditory_data))
auditory_data = list(map(lambda follow_timedelta: follow_timedelta.days, auditory_data))
auditory_data = np.array(auditory_data)

df = pd.DataFrame({
    "x": auditory_data,
})
fig = px.histogram(df, x="x")
fig.show() 

In [ ]:
len(auditory_data)

### Уникальность витуберов

Посмотрим, какая часть аудитории витубера смотрит только его 

In [ ]:
print(len(data.user_following))

In [ ]:
unique_users = list(filter(lambda fd: len(fd[1]) == 1, data.user_following.items()))
print(len(unique_users) / len(data.user_following) * 100, "%")

In [ ]:
unique_users2 = list(filter(lambda fd: len(fd[1]) == 2, data.user_following.items()))
print(len(unique_users2) / len(data.user_following) * 100, "%")

In [ ]:
unique_users3 = list(filter(lambda fd: len(fd[1]) >= 43, data.user_following.items()))
print(len(unique_users3) / len(data.user_following) * 100, "%")

In [ ]:
unique_users = list(filter(lambda fd: len(fd[1]) in range(1, 2), data.user_following.items()))
vtuber_unique_followers_counter = {}
for (user, following) in unique_users:
    for vtuber_name in following:
        vtuber_unique_followers_counter[vtuber_name] = vtuber_unique_followers_counter.get(vtuber_name, 0) + 1

vtuber_unique_followers_proportion = {}
for vtuber in data.vtuber_names:
    vtuber_unique_followers_proportion[vtuber] = 0
for (vtuber, unique_followers) in vtuber_unique_followers_counter.items():
    vtuber_unique_followers_proportion[vtuber] = unique_followers / len(data.vtuber_followers[vtuber]) * 100

# sorted(vtuber_unique_followers_proportion.items(), 
#        key=lambda el: el[1], 
#        reverse=True)

In [ ]:
df = pd.DataFrame({
    "Followers": list(map(lambda vtuber: len(data.vtuber_followers[vtuber]), data.vtuber_names)),
    "Unique followers (%)": list(map(lambda vtuber: vtuber_unique_followers_proportion[vtuber], data.vtuber_names)),
    "vtuber": data.vtuber_names
})
fig = px.scatter(df, x="Followers", y="Unique followers (%)", hover_name="vtuber")
fig.show()

### Кластеризация витуберов

#### Векторизация данных

In [ ]:
columns = data.user_following.keys()
user_id_to_column = {}
for (i, col) in zip(range(len(columns)), columns):
    user_id_to_column[col] = i

def vtuber2vec(vtuber: str) -> np.ndarray:
    result = np.zeros(len(data.user_following))
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None:
            result[user_id_to_column[follower.user.id]] = 1
    
    return result

In [ ]:
dataset = []
for vtuber in data.vtuber_names:
    dataset.append(vtuber2vec(vtuber))
dataset = np.array(dataset)

#### Кластеризация и отрисвка

##### Косинусное расстояние

In [ ]:
projection = sklearn.manifold.TSNE(metric="cosine", random_state=1).fit_transform(dataset)

In [ ]:
labels = sklearn.cluster.HDBSCAN(min_cluster_size=2, max_cluster_size=197, metric="cosine").fit_predict(X=dataset)
# labels = sklearn.cluster.DBSCAN(eps=0.757, min_samples=2, metric="cosine").fit_predict(X=dataset)

In [ ]:
data_frame = pd.DataFrame({
    "x": projection[:, 0],
    "y": projection[:, 1],
    "Кластер": labels,
    "Имя": data.vtuber_names
})
data_frame["Кластер"] = data_frame["Кластер"].astype(str) #convert to string

In [ ]:
fig = px.scatter(data_frame, 
                 x="x", 
                 y="y",
                 color="Кластер", 
                 hover_name="Имя")
fig.show()

In [ ]:
def cosine_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return 1 - np.dot(vec1, vec2) / np.sqrt(np.sum(vec1**2)) / np.sqrt(np.sum(vec2**2))

def manhattan_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return np.sum(np.abs(vec1 - vec2))

def simple_metric(vec1: np.ndarray, vec2: np.ndarray) -> float:
    inter = np.sum(vec1 * vec2)
    a = np.sum(vec1)
    b = np.sum(vec2)
    return 1 - inter / (a + b - inter)

def podsos(vec1: np.ndarray, vec2: np.ndarray) -> float:
    inter = np.sum(vec1 * vec2)
    a = np.sum(vec1)
    b = np.sum(vec2)
    return 1 - inter / (min(a, b))

In [ ]:
MATRIX = np.zeros((len(data.vtuber_names), len(data.vtuber_names)))
METRIC = cosine_metric

for i in range(len(data.vtuber_names)):
    print(f"{i})", data.vtuber_names[i])

    for j in range(len(data.vtuber_names)):
        vtuber1_vec = dataset[i]
        vtuber2_vec = dataset[j]
        distance = METRIC(vtuber1_vec, vtuber2_vec)
        MATRIX[i][j] = distance

In [ ]:
MAX_DISTANCE = 0.78
# MAX_DISTANCE = 0.6666666
EDGES = []
# IGNORED_VTUBERS = ['xkamysh', 'nyamuras']
IGNORED_VTUBERS = []

for i in range(len(data.vtuber_names)):
    for j in range(len(data.vtuber_names)):
        if MATRIX[i][j] < MAX_DISTANCE and i < j and data.vtuber_names[i] not in IGNORED_VTUBERS and data.vtuber_names[j] not in IGNORED_VTUBERS:
            EDGES.append((i, j))
print("Edges:", len(EDGES))


fig = go.Figure()

for (i, j) in EDGES:
    projection1 = projection[i]
    projection2 = projection[j]
    fig.add_trace(go.Scatter(x=[projection1[0], projection2[0]],
                             y=[projection1[1], projection2[1]],
                             mode="lines",
                             marker_color="#666688"))
colorscale = ['aggrnyl', 'agsunset', 'algae', 'amp', 'armyrose', 'balance',
    'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brbg', 'brwnyl', 
    'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'curl', 'darkmint', 'deep', 
    'delta', 'dense', 'earth', 'edge', 'electric', 'emrld', 'fall', 'geyser', 
    'gnbu', 'gray', 'greens', 'greys', 'haline', 'hot', 'hsv', 'ice', 'icefire', 
    'inferno', 'jet', 'magenta', 'magma', 'matter', 'mint', 'mrybm', 'mygbm', 
    'oranges', 'orrd', 'oryel', 'oxy', 'peach', 'phase', 'picnic', 'pinkyl', 
    'piyg', 'plasma', 'plotly3', 'portland', 'prgn', 'pubu', 'pubugn', 'puor', 
    'purd', 'purp', 'purples', 'purpor', 'rainbow', 'rdbu', 'rdgy', 'rdpu', 
    'rdylbu', 'rdylgn', 'redor', 'reds', 'solar', 'spectral', 'speed', 'sunset', 
    'sunsetdark', 'teal', 'tealgrn', 'tealrose', 'tempo', 'temps', 'thermal', 
    'tropic', 'turbid', 'turbo', 'twilight', 'viridis', 'ylgn', 'ylgnbu', 
    'ylorbr', 'ylorrd'][39]
fig.add_trace(go.Scatter(x=projection[:, 0], 
                         y=projection[:, 1],
                         marker=dict(
                            color=labels,
                            colorscale=colorscale,
                         ),
                         text=data.vtuber_names,
                         mode="markers"))
    
fig.show()

### Анализ количества фолловов у зрителей

#### Количество глобальных фолловов

In [ ]:
unique_users = []
used_logins = set()
for vtuber in data.vtuber_names:
    for follower_data in data.vtuber_followers[vtuber]:
        if follower_data.user is not None and follower_data.user.login not in used_logins:
            used_logins.add(follower_data.user.login)
            unique_users.append(follower_data.user)

In [ ]:
draw_data = unique_users
draw_data = list(map(lambda el: el.follows_count, draw_data))
draw_data = list(filter(lambda el: el < 100, draw_data))
df = pd.DataFrame({
    "Following": draw_data
})
fig = px.histogram(df, x="Following")
fig.show()

#### Количество локальных фолловов

In [ ]:
draw_data = data.user_following.items()
draw_data = list(map(lambda el: el[1], draw_data))
draw_data = list(map(lambda el: len(el), draw_data))
draw_data = list(filter(lambda el: el > 0, draw_data))
df = pd.DataFrame({
    "Following": draw_data
})
fig = px.histogram(df, x="Following")
fig.show()

#### Количество абсолютных уникалов среди фолловеров

In [ ]:
follows_count = []
abs_unique = []
for vtuber in data.vtuber_names:
    follows_count.append(len(data.vtuber_followers[vtuber]))
for vtuber in data.vtuber_names:
    vtuber_abs_unique = 0
    for follower in data.vtuber_followers[vtuber]:
        if follower.user is not None and follower.user.follows_count == 1:
            vtuber_abs_unique += 1
    abs_unique.append(vtuber_abs_unique)
abs_unique = np.array(abs_unique)
follows_count = np.array(follows_count)

df = pd.DataFrame({
    "Absolutely unique followers (%)": abs_unique / follows_count * 100,
    "Followers": follows_count,
    "Vtuber": vtuber_names,
})
fig = px.scatter(df, x="Followers", y="Absolutely unique followers (%)", hover_name="Vtuber")
fig.show()

## Аналитика сообщений

In [ ]:
@dataclass
class EmoteData:
    id: str


@dataclass
class MessageFragmentData:
    emote: Optional[EmoteData]
    text: Optional[str]
    mention: Optional[UserData]

    def as_text(self) -> str:
        if self.emote is not None:
            return f"<@{self.emote.id}>"
        elif self.text is not None:
            return self.text
        elif self.mention is not None:
            return f"@{self.mention.login}"
        else:
            return ""

    def as_plain_text(self) -> str:
        if self.emote is not None:
            return f"<@{self.emote.id}>"
        elif self.text is not None:
            return self.text
        elif self.mention is not None:
            return f"@{self.mention.login}"
        else:
            return ""


@dataclass
class MessageData:
    as_text: str
    as_plain_text: str
    fragments: List[MessageFragmentData]


@dataclass
class CommentData:
    id: str
    commenter: UserData
    contentOffsetSeconds: int
    message: MessageData


@dataclass
class VideoData:
    id: int
    title: str
    description: str
    created_at: datetime
    view_count: int
    comments: List[CommentData]


@dataclass
class VtuberVideosData:
    vtuber_names: List[str]
    vtuber_videos: Dict[str, List[VideoData]]

In [ ]:
def json_to_fragment_data(json_data) -> MessageFragmentData:
    return MessageFragmentData(emote=None if json_data['emote'] is None else EmoteData(id=json_data['emote']['id']),
                               text=json_data['text'],
                               mention=None if json_data['mention'] is None else json_to_user_data(json_data['mention']))


def fragment_data_to_json(fragment_data: MessageFragmentData):
    return {
        'emote': None if fragment_data.emote is None else {
            'id': fragment_data.emote.id
        },
        'text': fragment_data.text,
        'mention': None if fragment_data.mention is None else user_data_to_json(fragment_data.mention)
    }


def json_to_message_data(json_data) -> MessageData:
    as_text = ""
    as_plain_text = ""
    fragments = []

    for fragment_json in json_data['fragments']:
        fragment = json_to_fragment_data(fragment_json)
        as_text += fragment.as_text()
        as_plain_text += fragment.as_plain_text()
        fragments.append(fragment)

    return MessageData(as_text=as_text,
                       as_plain_text=as_plain_text,
                       fragments=fragments)


def message_data_to_json(message_data: MessageData):
    return {
        'fragments': list(map(fragment_data_to_json, message_data.fragments))
    }


def json_to_comment_data(json_data) -> CommentData:
    return CommentData(id=json_data['id'],
                       commenter=None if json_data['commenter'] is None else json_to_user_data(json_data['commenter']),
                       contentOffsetSeconds=int(json_data['contentOffsetSeconds']),
                       message=json_to_message_data(json_data['message']))


def user_data_to_json(user_data: UserData):
    return {
        'id': user_data.id,
        'login': user_data.login,
        'createdAt': None if user_data.created_at is None else str(user_data.created_at),
        'deletedAt': None if user_data.deleted_at is None else str(user_data.deleted_at),
        'follows': {
            'totalCount': str(user_data.follows_count)
        }
    }


def comment_data_to_json(comment_data: CommentData):
    return {
        'id': comment_data.id,
        'commenter': None if comment_data.commenter is None else user_data_to_json(comment_data.commenter),
        'contentOffsetSeconds': str(comment_data.contentOffsetSeconds),
        'message': message_data_to_json(comment_data.message)
    }


def json_to_video_data(json_data) -> VideoData:
    comments = []
    for comment in json_data['comments']['edges']:
        comments.append(json_to_comment_data(comment['node']))

    return VideoData(id=int(json_data['id']),
                     title=json_data['title'],
                     description=json_data['description'],
                     created_at=None if json_data['createdAt'] is None else datetime.fromisoformat(json_data['createdAt']),
                     view_count=int(json_data['viewCount']),
                     comments=comments)


def video_data_to_json(video_data: VideoData):
    return {
        'id': str(video_data.id),
        'title': video_data.title,
        'description': video_data.description,
        'createdAt': None if video_data.created_at is None else str(video_data.created_at),
        'viewCount': str(video_data.view_count),
        'comments': {
            'edges': list(map(
                lambda comment_data: {
                    'node': comment_data_to_json(comment_data)
                }, 
                video_data.comments))
        },
    }


def join_video_data(left: Optional[VideoData], right: VideoData) -> VideoData:
    if left is None:
        return right
    else:
        assert left.id == right.id

        comments = []
        comments.extend(left.comments)
        comments.extend(right.comments)

        return VideoData(id=left.id,
                         title=left.title,
                         description=left.description,
                         created_at=left.created_at,
                         view_count=left.view_count,
                         comments=comments)

In [ ]:
import requests
import time
import json
import pprint


API_URL = "https://gql.twitch.tv/gql"
API_CLIENT_ID = "kd1unb4b3q4t58fwlpcbzcbnm76a8fp"
API_REQUEST_DATA = """
query fetchVideoData($id: ID, $first: Int, $after: Cursor) {
    video(id: $id) {
        id
        title
        description
        createdAt
        viewCount
        comments(first: $first, after: $after) {
            edges {
                cursor
                node {
                    id
                    commenter {
                        id
                        login
                        createdAt
                        deletedAt
                        follows {
                            totalCount
                        }
                    }
                    contentOffsetSeconds
                    message {
                        fragments {
                            emote {
                                id
                            }
                            mention {
                                id
                                login
                                createdAt
                                deletedAt
                                follows {
                                    totalCount
                                }
                            }
                            text
                        }
                    }
                }
            }
            pageInfo {
                hasNextPage
            }
        }
    }
}
"""


def send_request(session: requests.Session, 
                 id: int, 
                 cursor: str,
                 log: bool,
                 repeat_times: int,
                 repeat_delay: float):
    for i in range(repeat_times):
        if i > 0:
            time.sleep(repeat_delay)

            if log:
                print("[LOG]", "Repeat last request")

        response = session.post(
            url=API_URL,
            json={
                'query': API_REQUEST_DATA,
                'variables': {
                    'id': str(id),
                    'first': 100,
                    'after': cursor
                }
            },
            headers={
                "Client-ID": API_CLIENT_ID
            })
        
        if response.status_code != 200:
            if log:
                print("[ERROR]", "Status code =", response.status_code)
            continue

        data = json.loads(response.text)
        if 'errors' in data.keys() and len(data['errors']) > 0:
            for error in data['errors']:
                if log:
                    print("[ERROR]", error)
            continue

        return response
    
    return None


def load_video_data(id: int, log: bool = False, repeat_times: int = 5, repeat_delay: float = 1.0) -> Optional[VideoData]:
    cache_path = f"./data/.temp/video/{id}.json"

    if os.path.exists(cache_path):
        if log:
            print("[LOG]", "Loading from the cache")
        with open(cache_path, 'r') as cache_file:
            return json_to_video_data(json.load(cache_file))

    session = requests.Session()
    cursor = None
    result = None

    while True:
        response = send_request(session, id, cursor, log, repeat_times, repeat_delay)

        if response is None:
            if log:
                print("[ERROR]", "Failed load", id)
            return None

        response_data = json.loads(response.text)
        result = join_video_data(result, json_to_video_data(response_data['data']['video']))
        

        if log:
            if len(result.comments) > 0:
                print(f"[LOG|{id}]", "Loaded", 
                      result.comments[-1].contentOffsetSeconds // 60 // 60, "hours", 
                      result.comments[-1].contentOffsetSeconds // 60 % 60, "minutes", 
                      result.comments[-1].contentOffsetSeconds % 60, "seconds")
            else:
                print(f"[LOG|{id}] Loaded ...")

        if len(response_data['data']['video']['comments']['edges']) == 0:
            break
        else:
            cursor = response_data['data']['video']['comments']['edges'][-1]['cursor']

        if not response_data['data']['video']['comments']['pageInfo']['hasNextPage'] or cursor == "":
            break

    if log:
        print('[LOG]', "caching")
    with open(cache_path, 'w') as cache_file:
        json.dump(video_data_to_json(result), cache_file, indent=4)

    return result
        


In [ ]:
API_REQUEST_DATA_2 = """
query fetchVideoData($login: String, $first: Int, $after: Cursor) {
    user(login: $login) {
        videos(first: $first, after: $after, type: ARCHIVE, sort: TIME) {
            totalCount
            pageInfo {
                hasNextPage
            }
            edges {
                cursor
                node {
                    id
                }
            }
        }
    }
}
"""


def send_request_2(session: requests.Session, 
                   login: str, 
                   cursor: str,
                   log: bool,
                   repeat_times: int,
                   repeat_delay: float):
    for i in range(repeat_times):
        if i > 0:
            time.sleep(repeat_delay)

            if log:
                print("[LOG]", "Repeat last request")

        response = session.post(
            url=API_URL,
            json={
                'query': API_REQUEST_DATA_2,
                'variables': {
                    'login': login,
                    'first': 20,
                    'after': cursor
                }
            },
            headers={
                "Client-ID": API_CLIENT_ID
            })
        
        if response.status_code != 200:
            if log:
                print("[ERROR]", "Status code =", response.status_code)
            continue

        data = json.loads(response.text)
        if 'errors' in data.keys() and len(data['errors']) > 0:
            for error in data['errors']:
                if log:
                    print("[ERROR]", error)
            continue

        return response
    
    return None


def load_streamer_videos(login: str, log: bool = False, repeat_times: int = 5, repeat_delay: float = 1.0, cacheonly: bool = False) -> Optional[List[int]]:
    result = []
    cache_path = f"./data/.temp/videos/{login}.json"

    if os.path.exists(cache_path):
        if log:
            print("[LOG]", "Loading from the cache")
        with open(cache_path, 'r') as cache_file:
            result = json.load(cache_file)

    if cacheonly:
        return result

    session = requests.Session()
    cursor = None

    while True:
        response = send_request_2(session, login, cursor, log, repeat_times, repeat_delay)
        assert response is not None

        response_data = json.loads(response.text)

        for video_data in response_data['data']['user']['videos']['edges']:
            cursor = video_data['cursor']
            id = int(video_data['node']['id'])

            if id not in result:
                result.append(id)
        
        if not response_data['data']['user']['videos']['pageInfo']['hasNextPage'] or cursor == "":
            break

    if log:
        print("[LOG]", "Caching result")
    with open(cache_path, 'w') as cache_file:
        json.dump(result, cache_file, indent=4)

    return result

In [ ]:
def load_vtuber_video(vtuber_login: str, video_id: int) -> Tuple[str, Optional[VideoData]]:
    assert load_video_data(video_id, log=True, repeat_times=10, repeat_delay=5.0) is not None


vtuber_videos = {}
with ThreadPoolExecutor(max_workers=64) as executor:
    futures = []

    for vtuber in vtuber_names:
        streamer_videos = load_streamer_videos(vtuber, log=True)
        assert streamer_videos is not None
        for video_id in streamer_videos:
            future = executor.submit(load_vtuber_video, vtuber_login=vtuber, video_id=video_id)
            futures.append(future)
    
    for future in as_completed(futures):
        future.result()

In [ ]:
# @dataclass
# class VtuberVideosData:
#     vtuber_names: List[str]
#     vtuber_videos: Dict[str, List[VideoData]]

vtuber_videos = {}
for vtuber in vtuber_names:
    vtuber_videos[vtuber] = []

    for video_id in load_streamer_videos(vtuber, log=True, cacheonly=True):
        video_data = load_video_data(video_id)
        vtuber_videos[vtuber].append(video_data)

In [ ]:
vvdata = VtuberVideosData(vtuber_names=vtuber_names,
                          vtuber_videos=vtuber_videos)

In [ ]:
messages_amount = 0
for vtuber in vvdata.vtuber_names:
    for video_data in vvdata.vtuber_videos[vtuber]:
        messages_amount += len(video_data.comments)
messages_amount

In [ ]:
commenters = set()
for vtuber in vvdata.vtuber_names:
    for video_data in vvdata.vtuber_videos[vtuber]:
        for comment in video_data.comments:
            if comment.commenter is not None:
                commenters.add(comment.commenter.id)
len(commenters)

In [ ]:
commenter_counters = {}
commenter_logins = {}
for commenter in commenters:
    commenter_counters[commenter] = 0
for vtuber in vvdata.vtuber_names:
    for video_data in vvdata.vtuber_videos[vtuber]:
        for comment in video_data.comments:
            if comment.commenter is not None:
                commenter_counters[comment.commenter.id] += 1
                commenter_logins[comment.commenter.id] = comment.commenter.login

In [ ]:
import random

FILTER_IDS = [100135110, 19264788, 1564983]


random.seed(6)
fig = go.Figure()


def not_ignore_comment_data(comment_data: CommentData) -> bool:
    return comment_data.commenter is None or comment_data.commenter.id not in FILTER_IDS


def add_plot(fig, streamer):
    max_index = len(vvdata.vtuber_videos[streamer])
    stream_data = vvdata.vtuber_videos[streamer][random.randint(0, max_index - 1)]
    print(streamer, ":", stream_data.id, stream_data.title, stream_data.created_at)
    
    plot_data = stream_data.comments
    plot_data = list(filter(not_ignore_comment_data, plot_data))
    plot_data = list(map(lambda comment: comment.contentOffsetSeconds // 600, plot_data))
    minutes = list(range(max(plot_data) + 1))
    counts = [0] * (max(plot_data) + 1)
    for el in plot_data:
        counts[el] += 1
    minutes = np.array(minutes)
    counts = np.array(counts)
    fig.add_trace(go.Scatter(x=minutes, 
                             y=counts,
                             mode='lines+markers',
                             name=streamer))

add_plot(fig, "xkamysh")
add_plot(fig, "qchaan_9")
add_plot(fig, "rera_seal")
add_plot(fig, "nyamuras")
add_plot(fig, "alaskanyan")
add_plot(fig, "yui2d")
add_plot(fig, "trixie_vox")
add_plot(fig, "akane_iwawwa")

fig.show()

In [ ]:
import random

random.seed(6)
fig = go.Figure()

def add_plot(fig, streamer):
    weights = []

    for stream_data in vvdata.vtuber_videos[streamer]:
        max_second = max(map(lambda comment: comment.contentOffsetSeconds, stream_data.comments))
        while len(weights) - 1 < max_second:
            weights.append(0)
        for i in range(max_second + 1):
            weights[i] += 1

    comments_counts = [0] * len(weights)
    for stream_data in vvdata.vtuber_videos[streamer]:
        for comment  in stream_data.comments:
            if not_ignore_comment_data(comment):
                comments_counts[comment.contentOffsetSeconds] += 1
    
    weights = np.array(weights)
    comments_counts = np.array(comments_counts)
    weighed_comments_counts = comments_counts / weights
    group_size = 300
    grouped_weighed_comments_counts = []
    for i in range(len(weighed_comments_counts)):
        if len(grouped_weighed_comments_counts) <= i // group_size:
            grouped_weighed_comments_counts.append(0)
        grouped_weighed_comments_counts[i // group_size] += weighed_comments_counts[i]
    times = np.array(range(len(weights)))

    fig.add_trace(go.Scatter(x=times, 
                             y=grouped_weighed_comments_counts,
                             mode='lines+markers',
                             name=streamer))

add_plot(fig, "xkamysh")
add_plot(fig, "qchaan_9")
add_plot(fig, "rera_seal")
add_plot(fig, "nyamuras")
add_plot(fig, "alaskanyan")
add_plot(fig, "yui2d")
add_plot(fig, "trixie_vox")
add_plot(fig, "akane_iwawwa")
add_plot(fig, "nubchann")
add_plot(fig, "planyach")
add_plot(fig, "banshameow")
add_plot(fig, "itsmoriko")
add_plot(fig, "severinasoda")
add_plot(fig, "snezha_mrr")
add_plot(fig, "conway671")
add_plot(fig, "ezdedus")

fig.show()


In [ ]:
GROUP_SIZE = 300 # 5 minutes
MAX_GROUP_COUNT = 36 # 3 hours


def get_streamer_chat_data(streamer: str) -> np.ndarray:
    weights = np.zeros(MAX_GROUP_COUNT * GROUP_SIZE)

    for stream_data in vvdata.vtuber_videos[streamer]:
        if len(stream_data.comments) == 0:
            max_second = 0
        else:
            max_second = max(map(lambda comment: comment.contentOffsetSeconds, stream_data.comments))
        for i in range(min(max_second + 1, MAX_GROUP_COUNT * GROUP_SIZE)):
            weights[i] += 1

    comments_counts = np.zeros(MAX_GROUP_COUNT * GROUP_SIZE)
    for stream_data in vvdata.vtuber_videos[streamer]:
        for comment in stream_data.comments:
            if not_ignore_comment_data(comment) and comment.contentOffsetSeconds < MAX_GROUP_COUNT * GROUP_SIZE:
                comments_counts[comment.contentOffsetSeconds] += 1
    
    weighed_comments_counts = comments_counts / weights
    
    result = []
    for el in np.array_split(weighed_comments_counts, MAX_GROUP_COUNT):
        result.append(np.sum(el))
    result = np.array(result)

    return result

In [ ]:
VTUBER_NAMES =  vvdata.vtuber_names
LEN = len(VTUBER_NAMES)
vtuber_average = np.zeros(LEN)
vtuber_std = np.zeros(LEN)
for (i, vtuber) in zip(range(LEN), VTUBER_NAMES):
    chat_data = get_streamer_chat_data(vtuber)
    vtuber_average[i] = np.average(chat_data)
    vtuber_std[i] = np.std(chat_data)
df = pd.DataFrame({
    "Chat_Average": vtuber_average,
    "Chat_Std": vtuber_std,
    "Followers": list(map(lambda vtuber: len(data.vtuber_followers[vtuber]), VTUBER_NAMES)),
    "Followers_Error_x": np.ones(LEN),
    "Vtubers": VTUBER_NAMES
})

In [ ]:
fig = px.scatter(df, 
                 x="Followers",
                 y="Chat_Average",
                 error_x="Followers_Error_x",
                 error_y="Chat_Std",
                 hover_name="Vtubers")
fig.show()